# NetworkX Lab: the Amazon co-purchase network
**Social Network Analytics (MBEL6275)**

In Gephi you *clicked* to get numbers. Here we *ask for them in code*, on the same 24,839 products and 73,981 links, so you can compare the two tools.

**How to use this notebook:** click a code cell, press **Shift + Enter** to run it, read the output, then move to the next cell. Run the cells in order, from the top.

## 0. Setup check

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sna_helpers import *      # our small helper functions (see sna_helpers.py)

print("NetworkX version:", nx.__version__)
print("pandas version:  ", pd.__version__)

## 1. A tiny network first: the edge list
A network is just a **list of pairs**. Each pair is a line (an *edge*) between two *nodes*.

In [ ]:
edge_list = [("Tony","Steve"), ("Tony","Natasha"), ("Steve","Natasha"),
             ("Natasha","Clint"), ("Tony","Thor"), ("Thor","Loki")]

toy = nx.Graph(edge_list)            # Graph = undirected
print("nodes:", list(toy.nodes))
print("edges:", list(toy.edges))
print("number of nodes / edges:", toy.number_of_nodes(), toy.number_of_edges())

Real data usually arrives as a table (a CSV file) with a `Source` and a `Target` column. `from_pandas_edgelist` turns that table into a network, and `to_pandas_edgelist` turns it back.

In [ ]:
df = pd.DataFrame(edge_list, columns=["Source", "Target"])
print(df)

same_toy = nx.from_pandas_edgelist(df, source="Source", target="Target")
print("\nSame network?", set(map(frozenset, same_toy.edges)) == set(map(frozenset, toy.edges)))

print("\nBack to a table:")
print(nx.to_pandas_edgelist(toy))

### Directed or undirected?
`nx.Graph` ignores direction. `nx.DiGraph` remembers it: an edge from Tony to Steve is **not** an edge from Steve to Tony.

In [ ]:
toy_d = nx.DiGraph(edge_list)

print("Tony -> Steve exists? ", toy_d.has_edge("Tony", "Steve"))
print("Steve -> Tony exists? ", toy_d.has_edge("Steve", "Tony"))
print("\nin-degree :", dict(toy_d.in_degree()))
print("out-degree:", dict(toy_d.out_degree()))
print("\nreciprocity (share of links that go both ways):", nx.reciprocity(toy_d))

toy_u = toy_d.to_undirected()       # drop the arrows
print("edges after dropping the arrows:", toy_u.number_of_edges())

**Your turn:** add the pair `("Steve", "Tony")` to `edge_list`, rebuild `toy_d`, and print the reciprocity again. What changed, and why?

### Centralities, shortest paths and density on the tiny network
The same functions will work on the big one.

In [ ]:
print("degree centrality     :", nx.degree_centrality(toy))
print("closeness centrality  :", {k: round(v,3) for k,v in nx.closeness_centrality(toy).items()})
print("betweenness centrality:", {k: round(v,3) for k,v in nx.betweenness_centrality(toy).items()})
print()
print("shortest path Clint -> Loki:", nx.shortest_path(toy, "Clint", "Loki"))
print("its length:", nx.shortest_path_length(toy, "Clint", "Loki"))
print("diameter:", nx.diameter(toy), "| average path length:", round(nx.average_shortest_path_length(toy),3))
print("density:", round(nx.density(toy),3))

## 2. Load the Amazon network
Two CSV files: one with the products (`Id`, `Label`, `group`) and one with the links (`Source`, `Target`). *(Gephi assignment Q1.)*

In [ ]:
nodes, edges, G = load_amazon()     # builds a DIRECTED graph (see sna_helpers.py)

print(nodes.head(), "\n")
print(edges.head(), "\n")
print(nodes["group"].value_counts(), "\n")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Type :", type(G).__name__)

## 3. Directed or undirected? Average degree and density
*(Gephi Q1 and Q3.)*

In [ ]:
print("reciprocity:", nx.reciprocity(G), "  <- how many links go both ways?")

U = G.to_undirected()                        # undirected copy
print("edges, directed  :", G.number_of_edges())
print("edges, undirected:", U.number_of_edges())

n, m = G.number_of_nodes(), G.number_of_edges()
print("\nlinks / nodes            :", round(m/n, 3), "   <- Gephi's 'Average Degree' on a directed network")
print("2 x links / nodes        :", round(2*m/n, 3), "   <- average of in-degree + out-degree")
print("density                  :", nx.density(G))

## 4. Degree centrality: in-degree, out-degree, total degree
*(Gephi Q7.)* In a directed `DiGraph`, `G.degree()` is **in + out**, the same as Gephi's `Degree` column.

In [ ]:
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())
tot_deg = dict(G.degree())

print("Most 'pointed-to' products (in-degree):")
display(top(in_deg, G, 5, "in_degree"))
print("Highest total degree:")
display(top(tot_deg, G, 5, "degree"))
print("Highest out-degree:")
display(top(out_deg, G, 5, "out_degree"))

In [ ]:
plt.figure(figsize=(7,3))
plt.hist(list(tot_deg.values()), bins=range(0, 60), color="grey", edgecolor="black")
plt.yscale("log"); plt.xlabel("Total degree"); plt.ylabel("Number of products (log scale)")
plt.title("Most products have a handful of links; a few have very many")
plt.show()

## 5. Shortest paths between two products
Pick two products by searching their titles. A *shortest path* is the route with the fewest links.

In [ ]:
a = find(G, "Brown Bear, Brown Bear")[0]
b = find(G, "Fight Club")[0]
print("From:", a[1], "(", a[2], ")")
print("To  :", b[1], "(", b[2], ")")

# Undirected: arrows ignored
path = nx.shortest_path(U, a[0], b[0])
print("\nUndirected path has", len(path)-1, "links:")
for p in path:
    print("  ", G.nodes[p]["label"][:50], "-", G.nodes[p]["group"])

# Directed: must follow the arrows
print("\nIs there a path that follows the arrows?", nx.has_path(G, a[0], b[0]))

In [ ]:
# Two products in the same neighbourhood, following the arrows
x = find(G, "Brown Bear, Brown Bear")[0][0]
reachable = nx.single_source_shortest_path_length(G, x)
print("Products reachable from 'Brown Bear' following arrows:", len(reachable) - 1, "out of", G.number_of_nodes() - 1)
print("Longest shortest path from it:", max(reachable.values()), "links")

## 6. Average path length and diameter
*(Gephi Q8.)* Exact answers on 25,000 nodes mean 25,000 searches. We use fast shortcuts and say so.

In [ ]:
print("Connected (ignoring arrows)?          ", nx.is_connected(U))
print("Strongly connected (following arrows)?", nx.is_strongly_connected(G))
print("Number of strongly connected components:", nx.number_strongly_connected_components(G))

In [ ]:
# UNDIRECTED: arrows ignored
print("Diameter (estimate, very fast):", nx.approximation.diameter(U))
apl = sample_avg_path(U, k=200)           # ~10 seconds
print("Average path length (200 random start points):", round(apl, 2))

In [ ]:
# DIRECTED: follow the arrows. Only pairs that have a path are counted.
diam_d, apl_d, share = directed_path_stats(G)
print("Directed diameter:", diam_d)
print("Directed average path length:", round(apl_d, 3))
print("Share of ordered pairs that are reachable:", round(share*100, 2), "%")

**Discuss:** why are the directed and undirected numbers so different? Which is the right one for a shopper who clicks 'customers also bought'?

> Exact undirected values (if you want to check): the diameter is 29 and the average path length is about 10.35. An exact run in NetworkX would take about 10 minutes, so we only use samples live.

## 7. Centralities: PageRank, betweenness, closeness
*(Gephi Q6, Q7 and Q8 / Q9.)*

In [ ]:
pr = nx.pagerank(G, alpha=0.85)            # directed graph: follows arrows
print("PageRank scores add up to:", round(sum(pr.values()), 4))
print("\nTop 10 by PageRank:")
display(top(pr, G, 10, "pagerank"))

In [ ]:
# Betweenness exactly would take hours here. k=100 uses 100 random start points: a fast estimate.
bt = nx.betweenness_centrality(U, k=100, seed=1)
print("Top 5 by betweenness (estimated):")
display(top(bt, G, 5, "betweenness"))

In [ ]:
# Closeness is a lot of work on the full network, so use one group: the Music products
music = [n for n, d in G.nodes(data=True) if d["group"] == "Music"]
M = U.subgraph(music)
cl = nx.closeness_centrality(M)
print("Music sub-network:", M.number_of_nodes(), "nodes, in", nx.number_connected_components(M), "pieces")
print("Top 5 by closeness inside the Music sub-network:")
display(top(cl, G, 5, "closeness"))

## 8. Community detection with Louvain
*(Gephi Q4 and Q5.)* Louvain starts with every product alone, merges them to raise **modularity**, and repeats.

In [ ]:
comms = nx.community.louvain_communities(U, seed=42)
Q = nx.community.modularity(U, comms)
sizes = sorted(map(len, comms), reverse=True)
print("Communities:", len(comms))
print("Modularity :", round(Q, 3))
print("Five largest sizes:", sizes[:5])

**Louvain has randomness.** Different seeds can give different answers. And the `resolution` setting changes how many communities you get (the same setting as in Gephi).

In [ ]:
table = louvain_table(U, resolutions=(0.5, 1.0, 2.0), seeds=range(5))
display(table.groupby("resolution").agg(communities_min=("communities","min"),
                                         communities_max=("communities","max"),
                                         modularity_mean=("modularity","mean")).round(3))

In [ ]:
# Community profiling: what is in each of the five largest communities?
top5 = sorted(comms, key=len, reverse=True)[:5]
rows = []
for i, c in enumerate(top5, 1):
    g = pd.Series([G.nodes[n]["group"] for n in c]).value_counts()
    rows.append({"community": i, "size": len(c), **g.to_dict()})
display(pd.DataFrame(rows).fillna(0).astype({"size": int}).set_index("community"))

In [ ]:
# Read some titles from each community. What do they have in common?
for i, c in enumerate(top5, 1):
    sample = sorted(c, key=lambda n: -tot_deg[n])[:4]
    print(i, [G.nodes[n]["label"][:30] for n in sample])

## 9. The top-5-community network: PageRank against degree
*(Gephi Q6 and Q7.)* Keep only the five largest communities, then rank products two ways.

In [ ]:
keep = set().union(*top5)
S = G.subgraph(keep).copy()                      # directed sub-network
print("Sub-network:", S.number_of_nodes(), "nodes,", S.number_of_edges(), "edges")

pr_s  = nx.pagerank(S)
deg_s = dict(S.degree())

a = top(pr_s, S, 10, "pagerank");  b = top(deg_s, S, 10, "degree")
display(a); display(b)

ids_pr  = set(sorted(pr_s,  key=pr_s.get,  reverse=True)[:10])
ids_deg = set(sorted(deg_s, key=deg_s.get, reverse=True)[:10])
print("Products in BOTH top-10 lists:", len(ids_pr & ids_deg))

In [ ]:
# Why do they differ? Look at the PageRank winner with the fewest links.
low = min(ids_pr, key=lambda n: S.in_degree(n))
print(S.nodes[low]["label"], "| in-links:", S.in_degree(low), "| PageRank:", round(pr_s[low], 5))
print("Who links to it?")
for p in S.predecessors(low):
    print("  ", S.nodes[p]["label"][:40], "| its own in-links:", S.in_degree(p), "| its out-links:", S.out_degree(p))

In [ ]:
# Diameter of the sub-network. Is it in one piece?
Us = S.to_undirected()
print("Pieces:", nx.number_connected_components(Us))
big = Us.subgraph(max(nx.connected_components(Us), key=len))
print("Largest piece:", big.number_of_nodes(), "nodes")
print("Its diameter (estimate):", nx.approximation.diameter(big))

## 10. Bridges
*(Gephi Q9.)* A bridge is a link whose removal disconnects the network.

In [ ]:
bridges = list(nx.bridges(U))
print("Bridges:", len(bridges), "of", U.number_of_edges(), "links =", round(100*len(bridges)/U.number_of_edges(), 1), "%")

deg1 = [n for n, d in U.degree() if d == 1]
print("Products with exactly one link:", len(deg1))

# how many pieces fall off when a bridge is removed?
cross = sum(1 for u, v in bridges if G.nodes[u]["group"] != G.nodes[v]["group"])
print("Bridges that join two different groups:", cross)

## 11. Your numbers against Gephi
Open **`02_compare_with_gephi.ipynb`** next, type in your Gephi values, and let the code tell you which ones match.